In [1]:
import sys
print(sys.version)


3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [3]:
!pip install torch torchvision torchaudio
!pip install transformers pandas numpy pillow scikit-learn tqdm matplotlib

  Using cached torch-2.9.1-cp312-cp312-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.24.1-cp312-cp312-win_amd64.whl.metadata (5.9 kB)
  Using cached torchaudio-2.9.1-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.9.1-cp312-cp312-win_amd64.whl (110.9 MB)
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.5/4.3 MB 840.2 kB/s eta 0:00:05
   ---- ----------------------------------- 0.5/4.3 MB 840.2 kB/s eta 0:00:05
   ---- ---------------

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Python312\\share'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2025.11.3-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.0 MB 2.1 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/12.0 MB 2.3 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/12.0 MB 2.0 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.0 MB 2.0 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/12.0 MB 1.5 MB/s eta 0:00:07
   ------ --------------------------------- 1.8/12.0 MB 1.5 MB/s eta 0:00:07
   ------ -----------------------

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python312\\Scripts\\hf.exe' -> 'C:\\Python312\\Scripts\\hf.exe.deleteme'


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import math
import pandas as pd
import numpy as np

from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split


c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BASE_DIR = "G:\My Drive"

DATASET_FOLDERS = [
    "olx_bikes_dataset",
    "olx_books_dataset",
    "olx_cars_dataset",
    "olx_cycle_dataset",
    "olx_flat_dataset",
    "olx_fridges_dataset",
    "olx_games_dataset",
    "olx_gamesentertainment_dataset",
    "olx_phones_dataset",
    "olx_printer_dataset",
    "olx_tv_dataset",
    "olx_washingmachine_dataset"
]


In [4]:
def read_csv_safe(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(
            path,
            encoding="latin1",
            encoding_errors="ignore"
        )



def load_all_datasets():
    dfs = []

    for folder in DATASET_FOLDERS:
        csv_path = os.path.join(BASE_DIR, folder, "dataset.csv")
        if not os.path.exists(csv_path):
            print(f"Missing: {csv_path}")
            continue

        df = read_csv_safe(csv_path)

        # Basic cleaning
        df = df.dropna(subset=["price", "title", "description"])
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        df = df[df["price"] > 0]

        df["category"] = folder.replace("olx_", "").replace("_dataset", "")
        dfs.append(df)

        print(f"Loaded {folder}: {len(df)} rows")

    full_df = pd.concat(dfs, ignore_index=True)
    return full_df


df = load_all_datasets()
print("\nTOTAL samples:", len(df))
df.head()


Loaded olx_bikes_dataset: 619 rows
Loaded olx_books_dataset: 302 rows
Loaded olx_cars_dataset: 440 rows
Loaded olx_cycle_dataset: 191 rows
Loaded olx_flat_dataset: 337 rows
Loaded olx_fridges_dataset: 500 rows
Loaded olx_games_dataset: 12 rows
Loaded olx_gamesentertainment_dataset: 497 rows
Loaded olx_phones_dataset: 299 rows
Loaded olx_printer_dataset: 263 rows
Loaded olx_tv_dataset: 480 rows
Loaded olx_washingmachine_dataset: 473 rows

TOTAL samples: 4413


,id,url,title,description,price,images,image_count,scraped_at,category
0,car_7d84be138c,https://www.olx.in/item/motorcycles-c81-used-b...,bajaj platina (2008),Motor bike - Motorcycles,15000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:19.5,bikes
1,car_b9d6dc8a84,https://www.olx.in/item/motorcycles-c81-used-t...,tvs star city plus (2025),Get Exciting Offers!! Star City Plus – Style T...,87805,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:30.8,bikes
2,car_6acd2594cf,https://www.olx.in/item/motorcycles-c81-used-y...,yamaha fz (2016),Explore used Yamaha bikes in Thaiyur. Find goo...,30000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:41.8,bikes
3,car_a9eaed64a0,https://www.olx.in/item/motorcycles-c81-used-b...,bajaj pulsar n160 (2024),Selling my Bajaj Pulsar N160 which is almost l...,145000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:51.4,bikes
4,car_2e40e67b7c,https://www.olx.in/item/motorcycles-c81-used-h...,hero xtreme 160r (2025),Get Exciting Offers Hero Xtreme 160R – Power. ...,129672,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,44:03.9,bikes


In [5]:
import numpy as np

# Remove invalid prices
df = df[df["price"].notna()]
df = df[df["price"] > 0]

# Log-transform price
df["log_price"] = np.log(df["price"].astype(float))

print("After cleaning:", len(df))
df[["price", "log_price"]].head()


After cleaning: 4413


,price,log_price
0,15000,9.615805
1,87805,11.382874
2,30000,10.308953
3,145000,11.884489
4,129672,11.772763


In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["category"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["category"]
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


Train: 3089
Val: 662
Test: 662


In [12]:
!pip install -q sentence-transformers

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python312\\Scripts\\torchfrtrace.exe' -> 'C:\\Python312\\Scripts\\torchfrtrace.exe.deleteme'


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [14]:
!pip install --upgrade pip setuptools wheel

# Core ML stack
!pip install \
torch torchvision torchaudio \
numpy pandas scipy \
scikit-learn tqdm matplotlib seaborn

# Text models (M1, M2)
!pip install \
sentence-transformers \
transformers \
tokenizers \
accelerate

# Vision models (M2, M3, M4)
!pip install \
timm \
opencv-python \
Pillow \
albumentations \
einops

# CLIP models (M3, M4)
!pip install \
open-clip-torch

# Tree models + ensemble utilities
!pip install \
xgboost \
lightgbm

# Utilities
!pip install \
pyyaml \
rich \
ipywidgets


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 13.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 29.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Python312\python.exe -m pip install --upgrade pip setuptools wheel


  Using cached torchvision-0.24.1-cp312-cp312-win_amd64.whl.metadata (5.9 kB)
  Using cached torchaudio-2.9.1-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
Using cached torchvision-0.24.1-cp312-cp312-win_amd64.whl (4.3 MB)
Using cached torchaudio-2.9.1-cp312-cp312-win_amd64.whl (665 kB)



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


  Using cached sentence_transformers-5.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
Using cached sentence_transformers-5.2.0-py3-none-any.whl (493 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.29.3
    Uninstalling huggingface-hub-0.29.3:
      Successfully uninstalled huggingface-hub-0.29.3
  Rolling back uninstall of huggingface-hub
  Moving to c:\python312\lib\site-packages\huggingface_hub-0.29.3.dist-info\
   from C:\Python312\Lib\site-packages\~uggingface_hub-0.29.3.dist-info
  Moving to c:\python312\lib\site-packages\huggingface_hub\
   from C:

...

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python312\\Scripts\\ftfy.exe' -> 'C:\\Python312\\Scripts\\ftfy.exe.deleteme'


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/72.0 MB 279.8 kB/s eta 0:04:16
   ---------------------------------------- 0.5/72.0 MB 279.8 kB/s eta 0:04:16
   ---------------------------------------- 0.5/72.0 MB 279.8 kB/s eta 0:04:16
   ---------------------------------------- 0.5/72.0 MB 279.8 kB/s eta 0:04:16
   ---------------------------------------- 0.5/72.0 MB 279.8 kB/s eta 0:04:16
   ---------------------------------------- 0


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   -

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Python312\\etc'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [7]:
import sys
print(sys.executable)


c:\Users\vishw\AppData\Local\Programs\Python\Python310\python.exe


In [8]:
!"c:\Users\vishw\AppData\Local\Programs\Python\Python310\python.exe" -m pip install sentence-transformers


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from sentence_transformers import SentenceTransformer
print("sentence-transformers works")



sentence-transformers works


In [10]:
train_df["text"] = train_df["title"] + " " + train_df["description"]
val_df["text"]   = val_df["title"]   + " " + val_df["description"]
test_df["text"]  = test_df["title"]  + " " + test_df["description"]

print(train_df["text"].iloc[0][:300])


2 bhk Baner-balewadi,1.15 cr, save upto 10 lakh pre launched offer A premium amenities project of balewadi - For Sale: Houses & Apartments


In [11]:
import os

os.environ["HF_HOME"] = os.path.join(os.getcwd(), "hf_cache")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(os.getcwd(), "hf_cache")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("HF cache set to:", os.environ["HF_HOME"])


HF cache set to: g:\My Drive\hf_cache


In [12]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print("Model loaded successfully")


c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vishw\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model loaded successfully


In [13]:
X_train_text = text_model.encode(
    train_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_val_text = text_model.encode(
    val_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_text = text_model.encode(
    test_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(X_train_text.shape, X_val_text.shape, X_test_text.shape)


Batches: 100%|██████████| 21/21 [00:07<00:00,  2.85it/s]

(3089, 384) (662, 384) (662, 384)


In [14]:
import torch

X_train = torch.tensor(X_train_text, dtype=torch.float32)
X_val   = torch.tensor(X_val_text, dtype=torch.float32)
X_test  = torch.tensor(X_test_text, dtype=torch.float32)

y_train = torch.tensor(train_df["log_price"].values, dtype=torch.float32)
y_val   = torch.tensor(val_df["log_price"].values, dtype=torch.float32)
y_test  = torch.tensor(test_df["log_price"].values, dtype=torch.float32)

print(X_train.shape, y_train.shape)


torch.Size([3089, 384]) torch.Size([3089])


In [15]:
import torch.nn as nn

class TextPIModel(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # mu1, mu2
        )

    def forward(self, x):
        return self.net(x)


In [16]:
def tube_loss(y, mu1, mu2, t=0.9, r=0.5):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)

    loss = torch.zeros_like(y)

    mask1 = y > upper
    loss[mask1] = t * (y[mask1] - upper[mask1])

    mask4 = y < lower
    loss[mask4] = t * (lower[mask4] - y[mask4])

    mask_mid = (~mask1) & (~mask4)
    mid = r * upper + (1 - r) * lower

    mask2 = mask_mid & (y >= mid)
    loss[mask2] = (1 - t) * (upper[mask2] - y[mask2])

    mask3 = mask_mid & (y < mid)
    loss[mask3] = (1 - t) * (y[mask3] - lower[mask3])

    return loss.mean()


In [17]:
model = TextPIModel(in_dim=X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 25
t = 0.9
r = 0.5

for epoch in range(epochs):
    model.train()

    out = model(X_train)
    mu1 = out[:, 0]
    mu2 = out[:, 1]

    loss = tube_loss(y_train, mu1, mu2, t=t, r=r)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")


Epoch 5/25 | Loss: 8.9358
Epoch 10/25 | Loss: 8.7664
Epoch 15/25 | Loss: 8.4593
Epoch 20/25 | Loss: 7.9324
Epoch 25/25 | Loss: 7.0909


In [18]:
model.eval()
with torch.no_grad():
    out = model(X_test)
    lower = torch.min(out[:, 0], out[:, 1])
    upper = torch.max(out[:, 0], out[:, 1])

    picp = ((y_test >= lower) & (y_test <= upper)).float().mean()
    mpiw = (upper - lower).mean()

print("M1 — Text Baseline (Tube Loss)")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M1 — Text Baseline (Tube Loss)
PICP: 0.003021148033440113
MPIW: 2.3698291778564453


In [19]:
y_mean = y_train.mean()
y_std  = y_train.std()

print("y_mean:", y_mean.item())
print("y_std :", y_std.item())

y_mean: 10.010866165161133
y_std : 2.5791304111480713


In [20]:
y_train_n = (y_train - y_mean) / y_std
y_val_n   = (y_val   - y_mean) / y_std
y_test_n  = (y_test  - y_mean) / y_std

In [21]:
model = TextPIModel(in_dim=X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30
t = 0.9
r = 0.5

for epoch in range(epochs):
    model.train()

    out = model(X_train)
    mu1 = out[:, 0]
    mu2 = out[:, 1]

    loss = tube_loss(y_train_n, mu1, mu2, t=t, r=r)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")


Epoch 5/30 | Loss: 0.5892
Epoch 10/30 | Loss: 0.4641
Epoch 15/30 | Loss: 0.3084
Epoch 20/30 | Loss: 0.1814
Epoch 25/30 | Loss: 0.1310
Epoch 30/30 | Loss: 0.1422


In [22]:
model.eval()
with torch.no_grad():
    out = model(X_test)

    lower_n = torch.min(out[:, 0], out[:, 1])
    upper_n = torch.max(out[:, 0], out[:, 1])

    # denormalize
    lower = lower_n * y_std + y_mean
    upper = upper_n * y_std + y_mean

    picp = ((y_test >= lower) & (y_test <= upper)).float().mean()
    mpiw = (upper - lower).mean()

print("M1 — Text Baseline (Tube Loss, normalized)")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M1 — Text Baseline (Tube Loss, normalized)
PICP: 0.9788519740104675
MPIW: 10.627857208251953


In [23]:
def eval_with_r(model, X, y, y_mean, y_std, r, t=0.9):
    model.eval()
    with torch.no_grad():
        out = model(X)
        lower_n = torch.min(out[:, 0], out[:, 1])
        upper_n = torch.max(out[:, 0], out[:, 1])

        lower = lower_n * y_std + y_mean
        upper = upper_n * y_std + y_mean

        picp = ((y >= lower) & (y <= upper)).float().mean().item()
        mpiw = (upper - lower).mean().item()

    return picp, mpiw

In [24]:
r_values = [0.1, 0.2, 0.3, 0.4, 0.5]
results = []

for r_val in r_values:
    picp, mpiw = eval_with_r(
        model, X_val, y_val, y_mean, y_std, r=r_val, t=0.9
    )
    results.append((r_val, picp, mpiw))
    print(f"r={r_val:.1f} | PICP={picp:.3f} | MPIW={mpiw:.3f}")

r=0.1 | PICP=0.970 | MPIW=10.569
r=0.2 | PICP=0.970 | MPIW=10.569
r=0.3 | PICP=0.970 | MPIW=10.569
r=0.4 | PICP=0.970 | MPIW=10.569
r=0.5 | PICP=0.970 | MPIW=10.569


In [25]:
best_r = 0.3  # ← replace with your chosen value

model = TextPIModel(in_dim=X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30
t = 0.9

for epoch in range(epochs):
    model.train()

    out = model(X_train)
    mu1 = out[:, 0]
    mu2 = out[:, 1]

    loss = tube_loss(y_train_n, mu1, mu2, t=t, r=best_r)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")


Epoch 5/30 | Loss: 0.6103
Epoch 10/30 | Loss: 0.5004
Epoch 15/30 | Loss: 0.3555
Epoch 20/30 | Loss: 0.2349
Epoch 25/30 | Loss: 0.1686
Epoch 30/30 | Loss: 0.1637


In [26]:
picp, mpiw = eval_with_r(
    model, X_test, y_test, y_mean, y_std, r=best_r, t=0.9
)

print("Final M1 (Text Baseline)")
print("r =", best_r)
print("PICP:", picp)
print("MPIW:", mpiw)

Final M1 (Text Baseline)
r = 0.3
PICP: 0.9501510858535767
MPIW: 10.752961158752441


M2

In [27]:
from PIL import Image
from torchvision import transforms
import os


In [28]:
import ast

def get_first_image(img_str):
    try:
        imgs = ast.literal_eval(img_str)
        if isinstance(imgs, list) and len(imgs) > 0:
            return imgs[1]
    except:
        pass
    return None

train_df["image_path"] = train_df["images"].apply(get_first_image)
val_df["image_path"]   = val_df["images"].apply(get_first_image)
test_df["image_path"]  = test_df["images"].apply(get_first_image)

train_df = train_df[train_df["image_path"].notna()]
val_df   = val_df[val_df["image_path"].notna()]
test_df  = test_df[test_df["image_path"].notna()]

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))


Train: 3089 Val: 662 Test: 662


In [29]:
from PIL import Image
from torchvision import transforms

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [44]:
import torch

class MultimodalDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ---- TEXT ----
        text = row["title"] + " " + row["description"]
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        # ---- IMAGE ----
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            img = img_transform(img)
        except:
            img = torch.zeros(3, 224, 224)

        # ---- TARGET (normalized log price) ----
        y = torch.tensor(row["log_price"], dtype=torch.float32)
        y = (y - y_mean) / y_std

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": img,
            "y": y
        }



In [32]:
!pip install timm

  Using cached timm-1.0.24-py3-none-any.whl.metadata (38 kB)
Using cached timm-1.0.24-py3-none-any.whl (2.6 MB)



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [34]:
!"c:\Users\vishw\AppData\Local\Programs\Python\Python310\python.exe" -m pip install timm


     ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
      --------------------------------------- 0.0/2.6 MB 245.8 kB/s eta 0:00:11
      --------------------------------------- 0.1/2.6 MB 328.2 kB/s eta 0:00:08
      --------------------------------------- 0.1/2.6 MB 328.2 kB/s eta 0:00:08
      --------------------------------------- 0.1/2.6 MB 328.2 kB/s eta 0:00:08
     - -------------------------------------- 0.1/2.6 MB 261.7 kB/s eta 0:00:10
     - -------------------------------------- 0.1/2.6 MB 261.7 kB/s eta 0:00:10
     - -------------------------------------- 0.1/2.6 MB 242.7 kB/s eta 0:00:11
     - -------------------------------------- 0.1/2.6 MB 257.2 kB/s eta 0:00:10
     - -------------------------------------- 0.1/2.6 MB 257.2 kB/s eta 0:00:10
     -- ------------------------------------- 0.1/2.6 MB 250.7 kB/s


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import timm
print("timm imported successfully")


timm imported successfully


In [45]:
from transformers import BertTokenizer, BertModel
import timm
import torch.nn as nn

# Text encoder
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

# Image encoder
effnet = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=0
)

print("Encoders loaded")

Encoders loaded


In [46]:
class MultimodalPIModel(nn.Module):
    def __init__(self, text_model, image_model):
        super().__init__()
        self.text_encoder = text_model
        self.image_encoder = image_model

        self.fc = nn.Sequential(
            nn.Linear(768 + image_model.num_features, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 2)  # mu1, mu2
        )

    def forward(self, input_ids, attention_mask, images):
        txt = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).pooler_output

        img = self.image_encoder(images)

        fused = torch.cat([txt, img], dim=1)
        return self.fc(fused)


In [47]:
from torch.utils.data import DataLoader

train_ds = MultimodalDataset(train_df, bert_tokenizer)
val_ds   = MultimodalDataset(val_df, bert_tokenizer)
test_ds  = MultimodalDataset(test_df, bert_tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False)

print("DataLoaders ready")



DataLoaders ready


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_m2 = MultimodalPIModel(bert_model, effnet).to(device)


Using device: cpu


In [49]:
# Freeze BERT & EfficientNet
for p in model_m2.text_encoder.parameters():
    p.requires_grad = False

for p in model_m2.image_encoder.parameters():
    p.requires_grad = False

optimizer = torch.optim.Adam(
    model_m2.fc.parameters(), lr=1e-3
)

print("Encoders frozen, optimizer ready")


Encoders frozen, optimizer ready


In [50]:
def tube_loss(y, mu1, mu2, t=0.9, r=0.3):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)

    loss = torch.zeros_like(y)

    mask1 = y > upper
    loss[mask1] = t * (y[mask1] - upper[mask1])

    mask4 = y < lower
    loss[mask4] = t * (lower[mask4] - y[mask4])

    mask_mid = (~mask1) & (~mask4)
    mid = r * upper + (1 - r) * lower

    mask2 = mask_mid & (y >= mid)
    loss[mask2] = (1 - t) * (upper[mask2] - y[mask2])

    mask3 = mask_mid & (y < mid)
    loss[mask3] = (1 - t) * (y[mask3] - lower[mask3])

    return loss.mean()


In [51]:
import os

# Your local Google Drive mirror
LOCAL_DRIVE_ROOT = r"G:\My Drive"

def fix_image_path(p):
    if not isinstance(p, str):
        return None

    # Colab → Local mapping
    if p.startswith("/content/drive/MyDrive/"):
        p = p.replace("/content/drive/MyDrive/", LOCAL_DRIVE_ROOT + os.sep)

    # Normalize for Windows
    p = os.path.normpath(p)

    return p if os.path.exists(p) else None


In [52]:
train_df["image_path"] = train_df["image_path"].apply(fix_image_path)
val_df["image_path"]   = val_df["image_path"].apply(fix_image_path)
test_df["image_path"]  = test_df["image_path"].apply(fix_image_path)

# Drop rows where image still not found
train_df = train_df[train_df["image_path"].notna()]
val_df   = val_df[val_df["image_path"].notna()]
test_df  = test_df[test_df["image_path"].notna()]

print("After fixing paths:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


After fixing paths:
Train: 2876
Val: 615
Test: 617


In [ ]:
model_m2.eval()

all_lower, all_upper, all_y = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        images = batch["image"].to(device)
        y_n = batch["y"].to(device)

        out = model_m2(input_ids, attention_mask, images)

        lower_n = torch.min(out[:, 0], out[:, 1])
        upper_n = torch.max(out[:, 0], out[:, 1])

        lower = lower_n * y_std + y_mean
        upper = upper_n * y_std + y_mean
        y = y_n * y_std + y_mean

        all_lower.append(lower.cpu())
        all_upper.append(upper.cpu())
        all_y.append(y.cpu())

lower = torch.cat(all_lower)
upper = torch.cat(all_upper)
y_true = torch.cat(all_y)

picp = ((y_true >= lower) & (y_true <= upper)).float().mean()
mpiw = (upper - lower).mean()

print("M2 — EfficientNet + BERT (Tube Loss)")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


In [54]:
import torch
from tqdm import tqdm

def cache_text_embeddings(df, tokenizer, model, device, max_len=128):
    model.eval()
    embeddings = []

    with torch.no_grad():
        for text in tqdm(df["title"] + " " + df["description"]):
            enc = tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(device)

            out = model(**enc).pooler_output
            embeddings.append(out.cpu())

    return torch.cat(embeddings)


In [55]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model = bert_model.to(device)

X_train_txt = cache_text_embeddings(train_df, bert_tokenizer, bert_model, device)
X_val_txt   = cache_text_embeddings(val_df, bert_tokenizer, bert_model, device)
X_test_txt  = cache_text_embeddings(test_df, bert_tokenizer, bert_model, device)

print("Text embeddings cached:", X_train_txt.shape)


100%|██████████| 617/617 [01:46<00:00,  5.79it/s]

Text embeddings cached: torch.Size([2876, 768])


In [56]:
def cache_image_embeddings(df, model, device):
    model.eval()
    embeddings = []

    with torch.no_grad():
        for p in tqdm(df["image_path"]):
            try:
                img = Image.open(p).convert("RGB")
                img = img_transform(img).unsqueeze(0).to(device)
            except:
                img = torch.zeros(1, 3, 224, 224).to(device)

            out = model(img)
            embeddings.append(out.cpu())

    return torch.cat(embeddings)


In [57]:
effnet = effnet.to(device)

X_train_img = cache_image_embeddings(train_df, effnet, device)
X_val_img   = cache_image_embeddings(val_df, effnet, device)
X_test_img  = cache_image_embeddings(test_df, effnet, device)

print("Image embeddings cached:", X_train_img.shape)


100%|██████████| 617/617 [11:56<00:00,  1.16s/it]

Image embeddings cached: torch.Size([2876, 1792])


In [58]:
X_train = torch.cat([X_train_txt, X_train_img], dim=1)
X_val   = torch.cat([X_val_txt,   X_val_img],   dim=1)
X_test  = torch.cat([X_test_txt,  X_test_img],  dim=1)

y_train_n = ((torch.tensor(train_df["log_price"].values) - y_mean) / y_std).float()
y_val_n   = ((torch.tensor(val_df["log_price"].values)   - y_mean) / y_std).float()
y_test_n  = ((torch.tensor(test_df["log_price"].values)  - y_mean) / y_std).float()

print(X_train.shape, y_train_n.shape)


torch.Size([2876, 2560]) torch.Size([2876])


In [59]:
class CachedDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [60]:
class CachedPIModel(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)


In [61]:
train_ds = CachedDataset(X_train, y_train_n)
val_ds   = CachedDataset(X_val, y_val_n)
test_ds  = CachedDataset(X_test, y_test_n)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)


In [62]:
model = CachedPIModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30
t = 0.9
r = 0.3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/30 | Loss: 0.1945
Epoch 2/30 | Loss: 0.1075
Epoch 3/30 | Loss: 0.1019
Epoch 4/30 | Loss: 0.0827
Epoch 5/30 | Loss: 0.0792
Epoch 6/30 | Loss: 0.0766
Epoch 7/30 | Loss: 0.0679
Epoch 8/30 | Loss: 0.0641
Epoch 9/30 | Loss: 0.0601
Epoch 10/30 | Loss: 0.0679
Epoch 11/30 | Loss: 0.0619
Epoch 12/30 | Loss: 0.0556
Epoch 13/30 | Loss: 0.0564
Epoch 14/30 | Loss: 0.0511
Epoch 15/30 | Loss: 0.0521
Epoch 16/30 | Loss: 0.0540
Epoch 17/30 | Loss: 0.0464
Epoch 18/30 | Loss: 0.0496
Epoch 19/30 | Loss: 0.0438
Epoch 20/30 | Loss: 0.0479
Epoch 21/30 | Loss: 0.0478
Epoch 22/30 | Loss: 0.0469
Epoch 23/30 | Loss: 0.0420
Epoch 24/30 | Loss: 0.0408
Epoch 25/30 | Loss: 0.0378
Epoch 26/30 | Loss: 0.0485
Epoch 27/30 | Loss: 0.0372
Epoch 28/30 | Loss: 0.0370
Epoch 29/30 | Loss: 0.0382
Epoch 30/30 | Loss: 0.0344


In [63]:
model.eval()
with torch.no_grad():
    out = model(X_test.to(device))

    lower = torch.min(out[:, 0], out[:, 1]) * y_std + y_mean
    upper = torch.max(out[:, 0], out[:, 1]) * y_std + y_mean
    y = y_test_n * y_std + y_mean

picp = ((y >= lower.cpu()) & (y <= upper.cpu())).float().mean()
mpiw = (upper - lower).mean()

print("M2 — Cached EfficientNet + BERT")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M2 — Cached EfficientNet + BERT
PICP: 0.7909238338470459
MPIW: 5.305780410766602


In [64]:
def eval_cached(model, X, y_n, r, t=0.9):
    model.eval()
    with torch.no_grad():
        out = model(X.to(device))
        lower = torch.min(out[:, 0], out[:, 1])
        upper = torch.max(out[:, 0], out[:, 1])

        # denormalize
        lower = lower * y_std + y_mean
        upper = upper * y_std + y_mean
        y = y_n * y_std + y_mean

        picp = ((y >= lower.cpu()) & (y <= upper.cpu())).float().mean().item()
        mpiw = (upper - lower).mean().item()

    return picp, mpiw


In [65]:
r_values = [0.3, 0.4, 0.5, 0.6]
for r_val in r_values:
    picp, mpiw = eval_cached(model, X_test, y_test_n, r=r_val, t=0.9)
    print(f"r={r_val:.1f} | PICP={picp:.3f} | MPIW={mpiw:.3f}")


r=0.3 | PICP=0.791 | MPIW=5.306
r=0.4 | PICP=0.791 | MPIW=5.306
r=0.5 | PICP=0.791 | MPIW=5.306
r=0.6 | PICP=0.791 | MPIW=5.306


In [66]:
best_r = 0.3      # keep same
t = 0.95          # increase coverage pressure

model = CachedPIModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=best_r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/30 | Loss: 0.1627
Epoch 2/30 | Loss: 0.0826
Epoch 3/30 | Loss: 0.0593
Epoch 4/30 | Loss: 0.0508
Epoch 5/30 | Loss: 0.0486
Epoch 6/30 | Loss: 0.0432
Epoch 7/30 | Loss: 0.0453
Epoch 8/30 | Loss: 0.0397
Epoch 9/30 | Loss: 0.0373
Epoch 10/30 | Loss: 0.0363
Epoch 11/30 | Loss: 0.0372
Epoch 12/30 | Loss: 0.0432
Epoch 13/30 | Loss: 0.0469
Epoch 14/30 | Loss: 0.0374
Epoch 15/30 | Loss: 0.0331
Epoch 16/30 | Loss: 0.0334
Epoch 17/30 | Loss: 0.0324
Epoch 18/30 | Loss: 0.0309
Epoch 19/30 | Loss: 0.0381
Epoch 20/30 | Loss: 0.0315
Epoch 21/30 | Loss: 0.0289
Epoch 22/30 | Loss: 0.0278
Epoch 23/30 | Loss: 0.0324
Epoch 24/30 | Loss: 0.0263
Epoch 25/30 | Loss: 0.0319
Epoch 26/30 | Loss: 0.0307
Epoch 27/30 | Loss: 0.0254
Epoch 28/30 | Loss: 0.0303
Epoch 29/30 | Loss: 0.0287
Epoch 30/30 | Loss: 0.0262


In [67]:
picp, mpiw = eval_cached(model, X_test, y_test_n, r=best_r, t=t)

print("M2 — Cached (higher t)")
print("t =", t)
print("PICP:", picp)
print("MPIW:", mpiw)


M2 — Cached (higher t)
t = 0.95
PICP: 0.8557536602020264
MPIW: 5.611708164215088


In [68]:
def tube_loss_delta(y, mu1, mu2, t=0.95, r=0.3, delta=0.05):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)

    loss = torch.zeros_like(y)

    mask1 = y > upper
    loss[mask1] = t * (y[mask1] - upper[mask1])

    mask4 = y < lower
    loss[mask4] = t * (lower[mask4] - y[mask4])

    mask_mid = (~mask1) & (~mask4)
    mid = r * upper + (1 - r) * lower

    mask2 = mask_mid & (y >= mid)
    loss[mask2] = (1 - t) * (upper[mask2] - y[mask2])

    mask3 = mask_mid & (y < mid)
    loss[mask3] = (1 - t) * (y[mask3] - lower[mask3])

    # width penalty
    width_penalty = delta * (upper - lower)

    return (loss + width_penalty).mean()


In [69]:
best_r = 0.3
t = 0.95
delta = 0.05   # start small

model = CachedPIModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = tube_loss_delta(
            y,
            out[:, 0],
            out[:, 1],
            t=t,
            r=best_r,
            delta=delta
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/30 | Loss: 0.2705
Epoch 2/30 | Loss: 0.1685
Epoch 3/30 | Loss: 0.1614
Epoch 4/30 | Loss: 0.1407
Epoch 5/30 | Loss: 0.1263
Epoch 6/30 | Loss: 0.1193
Epoch 7/30 | Loss: 0.1148
Epoch 8/30 | Loss: 0.1146
Epoch 9/30 | Loss: 0.1140
Epoch 10/30 | Loss: 0.1057
Epoch 11/30 | Loss: 0.0948
Epoch 12/30 | Loss: 0.0977
Epoch 13/30 | Loss: 0.1021
Epoch 14/30 | Loss: 0.0897
Epoch 15/30 | Loss: 0.0856
Epoch 16/30 | Loss: 0.0807
Epoch 17/30 | Loss: 0.0829
Epoch 18/30 | Loss: 0.0837
Epoch 19/30 | Loss: 0.0799
Epoch 20/30 | Loss: 0.0803
Epoch 21/30 | Loss: 0.0757
Epoch 22/30 | Loss: 0.0744
Epoch 23/30 | Loss: 0.0768
Epoch 24/30 | Loss: 0.0842
Epoch 25/30 | Loss: 0.0736
Epoch 26/30 | Loss: 0.0692
Epoch 27/30 | Loss: 0.0674
Epoch 28/30 | Loss: 0.0630
Epoch 29/30 | Loss: 0.0699
Epoch 30/30 | Loss: 0.0612


In [70]:
model.eval()
with torch.no_grad():
    out = model(X_test.to(device))

    lower = torch.min(out[:, 0], out[:, 1]) * y_std + y_mean
    upper = torch.max(out[:, 0], out[:, 1]) * y_std + y_mean
    y = y_test_n * y_std + y_mean

picp = ((y >= lower.cpu()) & (y <= upper.cpu())).float().mean()
mpiw = (upper - lower).mean()

print("M2 — Cached + width penalty")
print("t =", t, "delta =", delta)
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M2 — Cached + width penalty
t = 0.95 delta = 0.05
PICP: 0.6434359550476074
MPIW: 1.9024969339370728


In [71]:
def tube_loss(y, mu1, mu2, t=0.98, r=0.3):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)

    loss = torch.zeros_like(y)

    mask1 = y > upper
    loss[mask1] = t * (y[mask1] - upper[mask1])

    mask4 = y < lower
    loss[mask4] = t * (lower[mask4] - y[mask4])

    mask_mid = (~mask1) & (~mask4)
    mid = r * upper + (1 - r) * lower

    mask2 = mask_mid & (y >= mid)
    loss[mask2] = (1 - t) * (upper[mask2] - y[mask2])

    mask3 = mask_mid & (y < mid)
    loss[mask3] = (1 - t) * (y[mask3] - lower[mask3])

    return loss.mean()


In [72]:
best_r = 0.3
t = 0.98   # key change

model = CachedPIModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=best_r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/30 | Loss: 0.0821
Epoch 2/30 | Loss: 0.0441
Epoch 3/30 | Loss: 0.0319
Epoch 4/30 | Loss: 0.0304
Epoch 5/30 | Loss: 0.0348
Epoch 6/30 | Loss: 0.0264
Epoch 7/30 | Loss: 0.0272
Epoch 8/30 | Loss: 0.0221
Epoch 9/30 | Loss: 0.0207
Epoch 10/30 | Loss: 0.0207
Epoch 11/30 | Loss: 0.0204
Epoch 12/30 | Loss: 0.0240
Epoch 13/30 | Loss: 0.0212
Epoch 14/30 | Loss: 0.0196
Epoch 15/30 | Loss: 0.0183
Epoch 16/30 | Loss: 0.0187
Epoch 17/30 | Loss: 0.0177
Epoch 18/30 | Loss: 0.0180
Epoch 19/30 | Loss: 0.0195
Epoch 20/30 | Loss: 0.0175
Epoch 21/30 | Loss: 0.0191
Epoch 22/30 | Loss: 0.0198
Epoch 23/30 | Loss: 0.0192
Epoch 24/30 | Loss: 0.0191
Epoch 25/30 | Loss: 0.0166
Epoch 26/30 | Loss: 0.0155
Epoch 27/30 | Loss: 0.0165
Epoch 28/30 | Loss: 0.0207
Epoch 29/30 | Loss: 0.0166
Epoch 30/30 | Loss: 0.0156


In [73]:
model.eval()
with torch.no_grad():
    out = model(X_test.to(device))

    lower = torch.min(out[:, 0], out[:, 1]) * y_std + y_mean
    upper = torch.max(out[:, 0], out[:, 1]) * y_std + y_mean
    y = y_test_n * y_std + y_mean

picp = ((y >= lower.cpu()) & (y <= upper.cpu())).float().mean()
mpiw = (upper - lower).mean()

print("M2 — Cached (high t)")
print("t =", t)
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M2 — Cached (high t)
t = 0.98
PICP: 0.930307924747467
MPIW: 11.225118637084961


In [74]:
output_path = "cleaned_olx_dataset.csv"
df.to_csv(output_path, index=False)

print("Saved cleaned dataset to:", output_path)


Saved cleaned dataset to: cleaned_olx_dataset.csv


In [75]:
class ReferenceM2PIModel(nn.Module):
    def __init__(self, img_dim, txt_dim, dropout=0.3):
        super().__init__()

        common_dim = 512

        # Image projection
        self.image_projection = nn.Sequential(
            nn.Linear(img_dim, common_dim),
            nn.LayerNorm(common_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )

        # Text projection
        self.text_projection = nn.Sequential(
            nn.Linear(txt_dim, common_dim),
            nn.LayerNorm(common_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )

        # Cross-attention
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=common_dim,
            num_heads=8,
            dropout=dropout * 0.5,
            batch_first=True
        )

        # Fusion head (adapted for PI)
        self.fusion_head = nn.Sequential(
            nn.Linear(common_dim * 3, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(768, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(384, 192),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),

            nn.Linear(192, 2)  # μ1, μ2 (PI OUTPUT)
        )

    def forward(self, img_feat, txt_feat):
        # Project to common space
        img_p = self.image_projection(img_feat)
        txt_p = self.text_projection(txt_feat)

        # Cross-attention (image attends to text)
        attn_out, _ = self.cross_attention(
            img_p.unsqueeze(1),
            txt_p.unsqueeze(1),
            txt_p.unsqueeze(1)
        )

        attn_out = attn_out.squeeze(1)

        # Fusion
        fused = torch.cat([img_p, txt_p, attn_out], dim=1)

        return self.fusion_head(fused)


In [76]:
class CachedFusionDataset(torch.utils.data.Dataset):
    def __init__(self, X_img, X_txt, y):
        self.X_img = X_img
        self.X_txt = X_txt
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_img[idx],
            self.X_txt[idx],
            self.y[idx]
        )


In [77]:
train_ds = CachedFusionDataset(X_train_img, X_train_txt, y_train_n)
val_ds   = CachedFusionDataset(X_val_img,   X_val_txt,   y_val_n)
test_ds  = CachedFusionDataset(X_test_img,  X_test_txt,  y_test_n)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)


In [78]:
model = ReferenceM2PIModel(
    img_dim=X_train_img.shape[1],
    txt_dim=X_train_txt.shape[1],
    dropout=0.3
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

epochs = 40
t = 0.98
r = 0.3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for img, txt, y in train_loader:
        img = img.to(device)
        txt = txt.to(device)
        y = y.to(device)

        out = model(img, txt)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / len(train_loader):.4f}")


Epoch 1/40 | Loss: 0.0770
Epoch 2/40 | Loss: 0.0480
Epoch 3/40 | Loss: 0.0544
Epoch 4/40 | Loss: 0.0550
Epoch 5/40 | Loss: 0.0484
Epoch 6/40 | Loss: 0.0368
Epoch 7/40 | Loss: 0.0296
Epoch 8/40 | Loss: 0.0267
Epoch 9/40 | Loss: 0.0251
Epoch 10/40 | Loss: 0.0226
Epoch 11/40 | Loss: 0.0228
Epoch 12/40 | Loss: 0.0215
Epoch 13/40 | Loss: 0.0205
Epoch 14/40 | Loss: 0.0203
Epoch 15/40 | Loss: 0.0206
Epoch 16/40 | Loss: 0.0205
Epoch 17/40 | Loss: 0.0208
Epoch 18/40 | Loss: 0.0181
Epoch 19/40 | Loss: 0.0190
Epoch 20/40 | Loss: 0.0170
Epoch 21/40 | Loss: 0.0169
Epoch 22/40 | Loss: 0.0168
Epoch 23/40 | Loss: 0.0161
Epoch 24/40 | Loss: 0.0164
Epoch 25/40 | Loss: 0.0172
Epoch 26/40 | Loss: 0.0167
Epoch 27/40 | Loss: 0.0155
Epoch 28/40 | Loss: 0.0149
Epoch 29/40 | Loss: 0.0149
Epoch 30/40 | Loss: 0.0146
Epoch 31/40 | Loss: 0.0153
Epoch 32/40 | Loss: 0.0150
Epoch 33/40 | Loss: 0.0152
Epoch 34/40 | Loss: 0.0143
Epoch 35/40 | Loss: 0.0144
Epoch 36/40 | Loss: 0.0138
Epoch 37/40 | Loss: 0.0152
Epoch 38/4

In [79]:
model.eval()
with torch.no_grad():
    out = model(
        X_test_img.to(device),
        X_test_txt.to(device)
    )

    lower = torch.min(out[:, 0], out[:, 1]) * y_std + y_mean
    upper = torch.max(out[:, 0], out[:, 1]) * y_std + y_mean
    y = y_test_n * y_std + y_mean

picp = ((y >= lower.cpu()) & (y <= upper.cpu())).float().mean()
mpiw = (upper - lower).mean()

print("M2 — Reference-Architecture (PI)")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M2 — Reference-Architecture (PI)
PICP: 0.8995137810707092
MPIW: 14.098150253295898


M3

In [83]:
!"c:\Users\vishw\AppData\Local\Programs\Python\Python310\python.exe" -m pip install open_clip_torch


     ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.5 MB 1.1 MB/s eta 0:00:02
     ------ --------------------------------- 0.2/1.5 MB 2.5 MB/s eta 0:00:01
     -------------- ------------------------- 0.6/1.5 MB 4.0 MB/s eta 0:00:01
     ---------------------------------------  1.5/1.5 MB 8.2 MB/s eta 0:00:01
     ---------------------------------------- 1.5/1.5 MB 7.6 MB/s eta 0:00:00
     ---------------------------------------- 0.0/44.8 kB ? eta -:--:--
     ---------------------------------------- 44.8/44.8 kB 2.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [84]:
import open_clip
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader


In [85]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip_model, clip_preprocess, clip_tokenizer = open_clip.create_model_and_transforms(
    "ViT-L-14",
    pretrained="openai"
)

clip_model = clip_model.to(device)
clip_model.train()

print("CLIP ViT-L/14 loaded")


c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vishw\.cache\huggingface\hub\models--timm--vit_large_patch14_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\op

CLIP ViT-L/14 loaded


In [ ]:
import open_clip

CLIP_MODEL_NAME = "ViT-L-14"

clip_model, clip_preprocess, _ = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained="openai"
)

clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)

clip_model = clip_model.to(device)
for p in clip_model.parameters():
    p.requires_grad = False

clip_model.train()

print("CLIP model + tokenizer loaded correctly")


CLIP model + tokenizer loaded correctly


In [105]:
class CLIPDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ---- TEXT ----
        text = row["title"] + " " + row["description"]
        text_tokens = clip_tokenizer(text).squeeze(0)

        # ---- IMAGE ----
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            img = clip_preprocess(img)
        except:
            img = torch.zeros(3, 224, 224)

        # ---- TARGET ----
        y = torch.tensor(row["log_price"], dtype=torch.float32)
        y = (y - y_mean) / y_std

        return img, text_tokens, y


In [106]:
train_ds = CLIPDataset(train_df)
test_ds  = CLIPDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False)

print("CLIP DataLoaders rebuilt correctly")


CLIP DataLoaders rebuilt correctly


In [ ]:
class CLIPPIModel(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model

        # CLIP ViT-L/14 → 768 img + 768 text = 1536
        self.head = nn.Sequential(
            nn.Linear(1536, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, 2)  # μ1, μ2
        )

    def forward(self, images, texts):
    # ❄️ NO gradients for CLIP
        with torch.no_grad():
            img_feat = self.clip.encode_image(images)
            txt_feat = self.clip.encode_text(texts)

        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.head(fused)



In [114]:
# Freeze entire CLIP first
for p in clip_model.parameters():
    p.requires_grad = False

# Unfreeze last transformer blocks (safe fine-tuning)
for block in clip_model.visual.transformer.resblocks[-2:]:
    for p in block.parameters():
        p.requires_grad = True

for block in clip_model.transformer.resblocks[-2:]:
    for p in block.parameters():
        p.requires_grad = True

print("Last 2 CLIP blocks unfrozen")


Last 2 CLIP blocks unfrozen


In [115]:
model = CLIPPIModel(clip_model).to(device)

optimizer = torch.optim.AdamW([
    {"params": model.head.parameters(), "lr": 1e-3},
    {"params": filter(lambda p: p.requires_grad, clip_model.parameters()), "lr": 1e-5}
])


In [110]:
def tube_loss(y, mu1, mu2, t=0.95, r=0.5):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)

    loss = torch.zeros_like(y)

    loss[y > upper] = t * (y[y > upper] - upper[y > upper])
    loss[y < lower] = t * (lower[y < lower] - y[y < lower])

    mid = r * upper + (1 - r) * lower
    mask = (y >= lower) & (y <= upper)

    loss[mask & (y >= mid)] = (1 - t) * (upper[mask & (y >= mid)] - y[mask & (y >= mid)])
    loss[mask & (y < mid)]  = (1 - t) * (y[mask & (y < mid)] - lower[mask & (y < mid)])

    return loss.mean()


In [116]:
imgs, txts, y = next(iter(train_loader))
print("Image batch:", imgs.shape)   # (B, 3, 224, 224)
print("Text batch:", txts.shape)    # (B, 77)
print("Target:", y.shape)

Image batch: torch.Size([16, 3, 224, 224])
Text batch: torch.Size([16, 77])
Target: torch.Size([16])


In [117]:
epochs = 8
t = 0.95
r = 0.5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:,0], out[:,1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/8 | Loss: 0.0683
Epoch 2/8 | Loss: 0.0481
Epoch 3/8 | Loss: 0.0375
Epoch 4/8 | Loss: 0.0335
Epoch 5/8 | Loss: 0.0307
Epoch 6/8 | Loss: 0.0291
Epoch 7/8 | Loss: 0.0281
Epoch 8/8 | Loss: 0.0259


In [118]:
model.eval()
with torch.no_grad():
    lowers, uppers, ys = [], [], []

    for imgs, txts, y_n in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)

        out = model(imgs, txts)

        lower = torch.min(out[:,0], out[:,1]) * y_std + y_mean
        upper = torch.max(out[:,0], out[:,1]) * y_std + y_mean
        y = y_n * y_std + y_mean

        lowers.append(lower.cpu())
        uppers.append(upper.cpu())
        ys.append(y.cpu())

lower = torch.cat(lowers)
upper = torch.cat(uppers)
y = torch.cat(ys)

picp = ((y >= lower) & (y <= upper)).float().mean()
mpiw = (upper - lower).mean()

print("M3 — Fine-tuned OpenAI CLIP")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M3 — Fine-tuned OpenAI CLIP
PICP: 0.8865478038787842
MPIW: 5.026823997497559


In [119]:
t = 0.97
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:,0], out[:,1], t=t, r=0.5)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/3 | Loss: 0.0164
Epoch 2/3 | Loss: 0.0147
Epoch 3/3 | Loss: 0.0159


In [120]:
model.eval()
with torch.no_grad():
    lowers, uppers, ys = [], [], []

    for imgs, txts, y_n in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)

        out = model(imgs, txts)

        lower = torch.min(out[:,0], out[:,1]) * y_std + y_mean
        upper = torch.max(out[:,0], out[:,1]) * y_std + y_mean
        y = y_n * y_std + y_mean

        lowers.append(lower.cpu())
        uppers.append(upper.cpu())
        ys.append(y.cpu())

lower = torch.cat(lowers)
upper = torch.cat(uppers)
y = torch.cat(ys)

picp = ((y >= lower) & (y <= upper)).float().mean()
mpiw = (upper - lower).mean()

print("M3 — Fine-tuned OpenAI CLIP (final)")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M3 — Fine-tuned OpenAI CLIP (final)
PICP: 0.8995137810707092
MPIW: 5.718432903289795


M4

In [122]:
import open_clip
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLIP_MODEL_NAME = "ViT-H-14"

clip_model, clip_preprocess, _ = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained="laion2b_s32b_b79k"
)

clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)

clip_model = clip_model.to(device)
clip_model.train()

print("LAION CLIP ViT-H/14 loaded")


c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vishw\.cache\huggingface\hub\models--laion--CLIP-ViT-H-14-laion2B-s32B-b79K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


LAION CLIP ViT-H/14 loaded


In [123]:
class CLIPPIModel_M4(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model

        self.head = nn.Sequential(
            nn.Linear(2048, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(768, 384),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(384, 2)  # μ1, μ2
        )

    def forward(self, images, texts):
        img_feat = self.clip.encode_image(images)
        txt_feat = self.clip.encode_text(texts)

        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.head(fused)


In [124]:
# Freeze everything
for p in clip_model.parameters():
    p.requires_grad = False

# Unfreeze last 2 visual blocks
for block in clip_model.visual.transformer.resblocks[-2:]:
    for p in block.parameters():
        p.requires_grad = True

# Unfreeze last 2 text blocks
for block in clip_model.transformer.resblocks[-2:]:
    for p in block.parameters():
        p.requires_grad = True

print("Last 2 transformer blocks unfrozen (image + text)")


Last 2 transformer blocks unfrozen (image + text)


In [125]:
model_m4 = CLIPPIModel_M4(clip_model).to(device)

optimizer = torch.optim.AdamW([
    {"params": model_m4.head.parameters(), "lr": 1e-3},
    {"params": filter(lambda p: p.requires_grad, clip_model.parameters()), "lr": 5e-6}
])


In [126]:
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False)


In [127]:
epochs = 6
t = 0.97
r = 0.5

for epoch in range(epochs):
    model_m4.train()
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model_m4(imgs, txts)
        loss = tube_loss(y, out[:,0], out[:,1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/6 | Loss: 0.0547
Epoch 2/6 | Loss: 0.0367
Epoch 3/6 | Loss: 0.0301
Epoch 4/6 | Loss: 0.0261
Epoch 5/6 | Loss: 0.0244
Epoch 6/6 | Loss: 0.0233


In [128]:
model_m4.eval()

lowers, uppers, ys = [], [], []

with torch.no_grad():
    for imgs, txts, y_n in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)

        out = model_m4(imgs, txts)

        lower = torch.min(out[:,0], out[:,1]) * y_std + y_mean
        upper = torch.max(out[:,0], out[:,1]) * y_std + y_mean
        y = y_n * y_std + y_mean

        lowers.append(lower.cpu())
        uppers.append(upper.cpu())
        ys.append(y.cpu())

lower = torch.cat(lowers)
upper = torch.cat(uppers)
y = torch.cat(ys)

picp = ((y >= lower) & (y <= upper)).float().mean()
mpiw = (upper - lower).mean()

print("M4 — Fine-tuned LAION CLIP ViT-H/14")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


M4 — Fine-tuned LAION CLIP ViT-H/14
PICP: 0.9886547923088074
MPIW: 7.472214698791504


ENSEMBLE

In [129]:
# Ground-truth prices (denormalized)
y = y_test_n * y_std + y_mean
y = y.cpu()


In [131]:
[k for k in globals().keys() if "model" in k.lower()]


['AutoModel',
 'text_model',
 'TextPIModel',
 'model',
 'BertModel',
 'bert_model',
 'MultimodalPIModel',
 'model_m2',
 'CachedPIModel',
 'ReferenceM2PIModel',
 'clip_model',
 'CLIPPIModel',
 'CLIP_MODEL_NAME',
 'CLIPPIModel_M4',
 'model_m4']

In [132]:
# ---- FINAL MODEL ALIASES ----
model_m1 = TextPIModel    # M1: Text baseline PI model
model_m2 = model_m2       # M2: EfficientNet + BERT
model_m3 = model          # M3: OpenAI CLIP ViT-L/14
model_m4 = model_m4       # M4: LAION CLIP ViT-H/14


In [137]:
w3, w4 = 0.6, 0.4


In [139]:
# -------- M3 predictions --------
model_m3.eval()

l3_list, u3_list = [], []

with torch.no_grad():
    for imgs, txts, y_n in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)

        out = model_m3(imgs, txts)

        l = torch.min(out[:, 0], out[:, 1]) * y_std + y_mean
        u = torch.max(out[:, 0], out[:, 1]) * y_std + y_mean

        l3_list.append(l.cpu())
        u3_list.append(u.cpu())

l3 = torch.cat(l3_list)
u3 = torch.cat(u3_list)

print("✅ M3 predictions ready:", l3.shape)


✅ M3 predictions ready: torch.Size([617])


In [140]:
# -------- M4 predictions --------
model_m4.eval()

l4_list, u4_list = [], []

with torch.no_grad():
    for imgs, txts, y_n in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)

        out = model_m4(imgs, txts)

        l = torch.min(out[:, 0], out[:, 1]) * y_std + y_mean
        u = torch.max(out[:, 0], out[:, 1]) * y_std + y_mean

        l4_list.append(l.cpu())
        u4_list.append(u.cpu())

l4 = torch.cat(l4_list)
u4 = torch.cat(u4_list)

print("✅ M4 predictions ready:", l4.shape)


✅ M4 predictions ready: torch.Size([617])


In [141]:
w3, w4 = 0.6, 0.4

l_ens = w3 * l3 + w4 * l4
u_ens = w3 * u3 + w4 * u4

picp = ((y >= l_ens) & (y <= u_ens)).float().mean()
mpiw = (u_ens - l_ens).mean()

print("FINAL ENSEMBLE (M3 + M4)")
print("PICP:", picp.item())
print("MPIW:", mpiw.item())


FINAL ENSEMBLE (M3 + M4)
PICP: 0.9724473357200623
MPIW: 6.41970682144165


In [142]:
torch.save(
    {
        "y": y,
        "l3": l3, "u3": u3,
        "l4": l4, "u4": u4,
        "l_ens": l_ens, "u_ens": u_ens
    },
    "final_m3_m4_ensemble.pt"
)

print("✅ Final ensemble saved")


✅ Final ensemble saved


In [143]:
torch.save(
    {
        "model_state_dict": model_m4.state_dict(),
        "clip_backbone": "ViT-H-14",
        "pretrained": "laion2b_s32b_b79k",
        "t": 0.97,
        "r": 0.5
    },
    "M4_LAION_CLIP_model.pt"
)

print("✅ M4 model weights saved")


✅ M4 model weights saved


NEW

In [1]:
BASE_DIR = "G:\My Drive"

DATASET_FOLDERS = [
    "olx_bikes_dataset",
    "olx_books_dataset",
    "olx_cars_dataset",
    "olx_cycle_dataset",
    "olx_flat_dataset",
    "olx_fridges_dataset",
    "olx_games_dataset",
    "olx_gamesentertainment_dataset",
    "olx_laptop_dataset",
    "olx_mobile_dataset",
    "olx_phones_dataset",
    "olx_printer_dataset",
    "olx_tv_dataset",
    "olx_washingmachine_dataset"
]


In [2]:
import pandas as pd
import os

def read_csv_safe(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(
            path,
            encoding="latin1",
            encoding_errors="ignore"
        )



def load_all_datasets():
    dfs = []

    for folder in DATASET_FOLDERS:
        csv_path = os.path.join(BASE_DIR, folder, "dataset.csv")
        if not os.path.exists(csv_path):
            print(f"Missing: {csv_path}")
            continue

        df = read_csv_safe(csv_path)

        # Basic cleaning
        df = df.dropna(subset=["price", "title", "description"])
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        df = df[df["price"] > 0]

        df["category"] = folder.replace("olx_", "").replace("_dataset", "")
        dfs.append(df)

        print(f"Loaded {folder}: {len(df)} rows")

    full_df = pd.concat(dfs, ignore_index=True)
    return full_df


df = load_all_datasets()
print("\nTOTAL samples:", len(df))
df.head()


Loaded olx_bikes_dataset: 619 rows
Loaded olx_books_dataset: 302 rows
Loaded olx_cars_dataset: 440 rows
Loaded olx_cycle_dataset: 191 rows
Loaded olx_flat_dataset: 337 rows
Loaded olx_fridges_dataset: 500 rows
Loaded olx_games_dataset: 12 rows
Loaded olx_gamesentertainment_dataset: 497 rows
Loaded olx_laptop_dataset: 835 rows
Loaded olx_mobile_dataset: 356 rows
Loaded olx_phones_dataset: 299 rows
Loaded olx_printer_dataset: 263 rows
Loaded olx_tv_dataset: 480 rows
Loaded olx_washingmachine_dataset: 473 rows

TOTAL samples: 5604


,id,url,title,description,price,images,image_count,scraped_at,category
0,car_7d84be138c,https://www.olx.in/item/motorcycles-c81-used-b...,bajaj platina (2008),Motor bike - Motorcycles,15000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:19.5,bikes
1,car_b9d6dc8a84,https://www.olx.in/item/motorcycles-c81-used-t...,tvs star city plus (2025),Get Exciting Offers!! Star City Plus – Style T...,87805,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:30.8,bikes
2,car_6acd2594cf,https://www.olx.in/item/motorcycles-c81-used-y...,yamaha fz (2016),Explore used Yamaha bikes in Thaiyur. Find goo...,30000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:41.8,bikes
3,car_a9eaed64a0,https://www.olx.in/item/motorcycles-c81-used-b...,bajaj pulsar n160 (2024),Selling my Bajaj Pulsar N160 which is almost l...,145000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:51.4,bikes
4,car_2e40e67b7c,https://www.olx.in/item/motorcycles-c81-used-h...,hero xtreme 160r (2025),Get Exciting Offers Hero Xtreme 160R – Power. ...,129672,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,44:03.9,bikes


In [3]:
df = load_all_datasets()
print("Total samples (old + new):", len(df))


Loaded olx_bikes_dataset: 619 rows
Loaded olx_books_dataset: 302 rows
Loaded olx_cars_dataset: 440 rows
Loaded olx_cycle_dataset: 191 rows
Loaded olx_flat_dataset: 337 rows
Loaded olx_fridges_dataset: 500 rows
Loaded olx_games_dataset: 12 rows
Loaded olx_gamesentertainment_dataset: 497 rows
Loaded olx_laptop_dataset: 835 rows
Loaded olx_mobile_dataset: 356 rows
Loaded olx_phones_dataset: 299 rows
Loaded olx_printer_dataset: 263 rows
Loaded olx_tv_dataset: 480 rows
Loaded olx_washingmachine_dataset: 473 rows
Total samples (old + new): 5604


In [4]:
print(df.columns)


Index(['id', 'url', 'title', 'description', 'price', 'images', 'image_count',
       'scraped_at', 'category'],
      dtype='object')


In [5]:
import numpy as np

df = df.dropna(subset=["price", "title", "description"])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df[df["price"] > 0]

df["log_price"] = np.log(df["price"].astype(float))

print("After cleaning:", len(df))


After cleaning: 5604


In [6]:
df["category"].value_counts()

category
laptop                835
bikes                 619
fridges               500
gamesentertainment    497
tv                    480
washingmachine        473
cars                  440
mobile                356
flat                  337
books                 302
phones                299
printer               263
cycle                 191
games                  12
Name: count, dtype: int64

In [7]:
category_dfs = {}

for cat in sorted(df["category"].unique()):
    cat_df = df[df["category"] == cat].copy()
    category_dfs[cat] = cat_df
    print(f"{cat}: {len(cat_df)} samples")


bikes: 619 samples
books: 302 samples
cars: 440 samples
cycle: 191 samples
flat: 337 samples
fridges: 500 samples
games: 12 samples
gamesentertainment: 497 samples
laptop: 835 samples
mobile: 356 samples
phones: 299 samples
printer: 263 samples
tv: 480 samples
washingmachine: 473 samples


In [8]:
category_stats = {}

for cat, cat_df in category_dfs.items():
    prices = cat_df["price"].values
    log_prices = np.log(prices)

    category_stats[cat] = {
        "count": len(prices),
        "mean_price": float(prices.mean()),
        "median_price": float(np.median(prices)),
        "std_price": float(prices.std()),
        "min_price": float(prices.min()),
        "max_price": float(prices.max()),
        "log_mean": float(log_prices.mean()),
        "log_std": float(log_prices.std()),
    }


In [9]:
for cat, s in category_stats.items():
    print(f"\nCategory: {cat}")
    print(f"  Count: {s['count']}")
    print(f"  Mean price: {s['mean_price']:.2f}")
    print(f"  Std price: {s['std_price']:.2f}")
    print(f"  Min–Max: {s['min_price']:.2f} – {s['max_price']:.2f}")



Category: bikes
  Count: 619
  Mean price: 106404.05
  Std price: 119747.24
  Min–Max: 1.00 – 1550000.00

Category: books
  Count: 302
  Mean price: 1774.03
  Std price: 3010.87
  Min–Max: 25.00 – 15000.00

Category: cars
  Count: 440
  Mean price: 967705.53
  Std price: 1152914.47
  Min–Max: 435.00 – 11950000.00

Category: cycle
  Count: 191
  Mean price: 12583.22
  Std price: 58970.72
  Min–Max: 300.00 – 800000.00

Category: flat
  Count: 337
  Mean price: 7356195.75
  Std price: 6502685.16
  Min–Max: 3.00 – 40000000.00

Category: fridges
  Count: 500
  Mean price: 10985.81
  Std price: 15262.07
  Min–Max: 299.00 – 150000.00

Category: games
  Count: 12
  Mean price: 20858.17
  Std price: 12235.56
  Min–Max: 1800.00 – 40000.00

Category: gamesentertainment
  Count: 497
  Mean price: 16453.04
  Std price: 17499.08
  Min–Max: 1.00 – 100000.00

Category: laptop
  Count: 835
  Mean price: 29274.27
  Std price: 28576.81
  Min–Max: 17.00 – 250000.00

Category: mobile
  Count: 356
  Mean p

In [159]:
import os, json

out_path = os.path.join(os.getcwd(), "category_price_stats.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(category_stats, f, indent=2)

print("Saved category-wise statistics to:", out_path)


Saved category-wise statistics to: g:\My Drive\category_price_stats.json


In [10]:
from sklearn.model_selection import train_test_split

category_splits = {}

RANDOM_STATE = 42

for cat, cat_df in category_dfs.items():
    # First split: train vs temp
    train_df, temp_df = train_test_split(
        cat_df,
        test_size=0.30,
        random_state=RANDOM_STATE
    )

    # Second split: val vs test
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_STATE
    )

    category_splits[cat] = {
        "train": train_df.reset_index(drop=True),
        "val": val_df.reset_index(drop=True),
        "test": test_df.reset_index(drop=True),
    }

    print(
        f"{cat}: "
        f"train={len(train_df)}, "
        f"val={len(val_df)}, "
        f"test={len(test_df)}"
    )


bikes: train=433, val=93, test=93
books: train=211, val=45, test=46
cars: train=308, val=66, test=66
cycle: train=133, val=29, test=29
flat: train=235, val=51, test=51
fridges: train=350, val=75, test=75
games: train=8, val=2, test=2
gamesentertainment: train=347, val=75, test=75
laptop: train=584, val=125, test=126
mobile: train=249, val=53, test=54
phones: train=209, val=45, test=45
printer: train=184, val=39, test=40
tv: train=336, val=72, test=72
washingmachine: train=331, val=71, test=71


In [11]:
for cat, splits in category_splits.items():
    print(f"\nCategory: {cat}")
    for split_name, split_df in splits.items():
        print(
            f"  {split_name}: "
            f"mean={split_df['price'].mean():.2f}, "
            f"std={split_df['price'].std():.2f}"
        )



Category: bikes
  train: mean=100822.97, std=93196.30
  val: mean=116573.87, std=188268.73
  test: mean=122219.28, std=140519.77

Category: books
  train: mean=1615.80, std=2633.92
  val: mean=2665.89, std=4446.51
  test: mean=1627.35, std=2862.91

Category: cars
  train: mean=957126.68, std=1145830.80
  val: mean=931074.88, std=1089854.41
  test: mean=1053704.17, std=1264711.58

Category: cycle
  train: mean=13257.11, std=69347.87
  val: mean=11865.52, std=27714.00
  test: mean=10210.31, std=16542.12

Category: flat
  train: mean=7161310.27, std=6310525.36
  val: mean=8609899.47, std=8962999.77
  test: mean=7000493.76, std=4047691.80

Category: fridges
  train: mean=10947.18, std=14555.35
  val: mean=10648.75, std=12898.26
  test: mean=11503.16, std=20197.24

Category: games
  train: mean=20187.38, std=12732.74
  val: mean=30999.50, std=8484.57
  test: mean=13400.00, std=16404.88

Category: gamesentertainment
  train: mean=16573.27, std=17150.53
  val: mean=17173.72, std=19707.98
  t

In [14]:
import os

os.makedirs("category_splits", exist_ok=True)

for cat, splits in category_splits.items():
    for split_name, split_df in splits.items():
        path = f"category_splits/{cat}_{split_name}.csv"
        split_df.to_csv(path, index=False)

print("Saved category-wise train/val/test splits")


Saved category-wise train/val/test splits


In [12]:
import numpy as np

category_norm = {}

for cat, splits in category_splits.items():
    train_df = splits["train"]

    log_prices = np.log(train_df["price"].values)

    category_norm[cat] = {
        "log_mean": float(log_prices.mean()),
        "log_std": float(log_prices.std())
    }

    print(
        f"{cat}: log_mean={category_norm[cat]['log_mean']:.3f}, "
        f"log_std={category_norm[cat]['log_std']:.3f}"
    )


bikes: log_mean=11.179, log_std=1.030
books: log_mean=6.536, log_std=1.308
cars: log_mean=13.371, log_std=0.938
cycle: log_mean=8.431, log_std=1.072
flat: log_mean=15.291, log_std=1.763
fridges: log_mean=8.828, log_std=0.999
games: log_mean=9.671, log_std=0.766
gamesentertainment: log_mean=8.947, log_std=1.531
laptop: log_mean=9.869, log_std=1.037
mobile: log_mean=9.876, log_std=0.937
phones: log_mean=9.723, log_std=1.021
printer: log_mean=9.067, log_std=1.440
tv: log_mean=8.813, log_std=1.440
washingmachine: log_mean=8.762, log_std=0.872


In [13]:
for cat, splits in category_splits.items():
    mu = category_norm[cat]["log_mean"]
    sigma = category_norm[cat]["log_std"]

    for split_name, split_df in splits.items():
        split_df["log_price_norm"] = (
            np.log(split_df["price"].values) - mu
        ) / sigma


In [14]:
for cat, splits in category_splits.items():
    train_vals = splits["train"]["log_price_norm"]
    val_vals   = splits["val"]["log_price_norm"]
    test_vals  = splits["test"]["log_price_norm"]

    print(f"\n{cat}")
    print(f"  train mean/std: {train_vals.mean():.3f}, {train_vals.std():.3f}")
    print(f"  val   mean/std: {val_vals.mean():.3f}, {val_vals.std():.3f}")
    print(f"  test  mean/std: {test_vals.mean():.3f}, {test_vals.std():.3f}")



bikes
  train mean/std: 0.000, 1.001
  val   mean/std: -0.051, 1.060
  test  mean/std: 0.156, 0.828

books
  train mean/std: -0.000, 1.002
  val   mean/std: 0.094, 1.285
  test  mean/std: -0.094, 1.047

cars
  train mean/std: 0.000, 1.002
  val   mean/std: -0.005, 0.892
  test  mean/std: 0.081, 0.916

cycle
  train mean/std: 0.000, 1.004
  val   mean/std: 0.154, 0.931
  test  mean/std: 0.161, 0.966

flat
  train mean/std: -0.000, 1.002
  val   mean/std: -0.081, 1.235
  test  mean/std: 0.047, 1.077

fridges
  train mean/std: 0.000, 1.001
  val   mean/std: -0.085, 1.132
  test  mean/std: -0.127, 1.129

games
  train mean/std: -0.000, 1.069
  val   mean/std: 0.850, 0.362
  test  mean/std: -1.122, 2.428

gamesentertainment
  train mean/std: -0.000, 1.001
  val   mean/std: -0.175, 1.299
  test  mean/std: -0.061, 0.914

laptop
  train mean/std: -0.000, 1.001
  val   mean/std: 0.065, 1.028
  test  mean/std: -0.071, 0.968

mobile
  train mean/std: -0.000, 1.002
  val   mean/std: -0.014, 0.916

In [18]:
import json, os

norm_path = os.path.join(BASE_DIR, "category_normalization.json")

with open(norm_path, "w", encoding="utf-8") as f:
    json.dump(category_norm, f, indent=2)

print("Saved category-wise normalization to:", norm_path)


Saved category-wise normalization to: G:\My Drive\category_normalization.json


In [15]:
TARGET_CATEGORIES = sorted(category_dfs.keys())
print(TARGET_CATEGORIES)

['bikes', 'books', 'cars', 'cycle', 'flat', 'fridges', 'games', 'gamesentertainment', 'laptop', 'mobile', 'phones', 'printer', 'tv', 'washingmachine']


In [17]:
import open_clip
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

CLIP_MODEL_NAME = "ViT-L-14"

clip_model, clip_preprocess, _ = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained="openai"
)

clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)

clip_model = clip_model.to(device)

# Freeze CLIP
for p in clip_model.parameters():
    p.requires_grad = False

clip_model.eval()   # ✅ IMPORTANT

print("CLIP model + tokenizer loaded correctly")


CLIP model + tokenizer loaded correctly


In [18]:
class CLIPPIModel(nn.Module):
    def __init__(self, clip_model, embed_dim=768):
        super().__init__()
        self.clip = clip_model
        self.head = nn.Sequential(
            nn.Linear(embed_dim * 2, 512),
            nn.ReLU(),
            nn.Linear(512, 2)   
        )

    def forward(self, images, texts):
    # ❄️ NO gradients for CLIP
        with torch.no_grad():
            img_feat = self.clip.encode_image(images)
            txt_feat = self.clip.encode_text(texts)

        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.head(fused)




In [19]:
def tube_loss(y, mu1, mu2, t=0.9, r=0.5):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)

    loss = torch.zeros_like(y)

    mask1 = y > upper
    loss[mask1] = t * (y[mask1] - upper[mask1])

    mask4 = y < lower
    loss[mask4] = t * (lower[mask4] - y[mask4])

    mask_mid = (~mask1) & (~mask4)
    mid = r * upper + (1 - r) * lower

    mask2 = mask_mid & (y >= mid)
    loss[mask2] = (1 - t) * (upper[mask2] - y[mask2])

    mask3 = mask_mid & (y < mid)
    loss[mask3] = (1 - t) * (y[mask3] - lower[mask3])

    return loss.mean()


In [20]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import ast

class CLIPPriceDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # TEXT
        text = (row["title"] or "") + " " + (row["description"] or "")
        text_tokens = self.tokenizer(text)[0]

        # IMAGE
        img_paths = ast.literal_eval(row["images"])
        img_path = img_paths[1]

        try:
            image = Image.open(img_path).convert("RGB")
            image = self.image_transform(image)
        except Exception:
            image = torch.zeros(3, 224, 224)

        # TARGET
        y = torch.tensor(row["log_price_norm"], dtype=torch.float32)

        return image, text_tokens, y


In [21]:
category_datasets = {}

for cat, splits in category_splits.items():
    train_ds = CLIPPriceDataset(
        splits["train"],
        tokenizer=clip_tokenizer,
        image_transform=clip_preprocess
    )

    val_ds = CLIPPriceDataset(
        splits["val"],
        tokenizer=clip_tokenizer,
        image_transform=clip_preprocess
    )

    test_ds = CLIPPriceDataset(
        splits["test"],
        tokenizer=clip_tokenizer,
        image_transform=clip_preprocess
    )

    category_datasets[cat] = {
        "train": train_ds,
        "val": val_ds,
        "test": test_ds
    }

print("Category datasets rebuilt")


Category datasets rebuilt


In [22]:
img, txt, y = category_datasets[TARGET_CATEGORIES[0]]["train"][0]

print("Image:", img.shape)      # (3, 224, 224)
print("Text:", txt.shape)       # (77,)
print("Target:", y.shape)       # ()


Image: torch.Size([3, 224, 224])
Text: torch.Size([77])
Target: torch.Size([])


In [23]:
from torch.utils.data import DataLoader

dl = DataLoader(category_datasets[TARGET_CATEGORIES[0]]["train"], batch_size=8)
imgs, txts, ys = next(iter(dl))

print("Batch image:", imgs.shape)   # (8, 3, 224, 224)
print("Batch text:", txts.shape)    # (8, 77)
print("Batch y:", ys.shape)         # (8,)


Batch image: torch.Size([8, 3, 224, 224])
Batch text: torch.Size([8, 77])
Batch y: torch.Size([8])


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import Adam

results_m3 = {}

for cat in TARGET_CATEGORIES:
    print(f"\n===== Training M3 for category: {cat} =====")

    train_ds = category_datasets[cat]["train"]
    val_ds   = category_datasets[cat]["val"]
    test_ds  = category_datasets[cat]["test"]

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=16)
    test_loader  = DataLoader(test_ds, batch_size=16)

    model = CLIPPIModel(clip_model).to(device)

    optimizer = Adam(model.parameters(), lr=1e-4)

    t = 0.95   # category-wise later we can tune this
    r = 0.5

    # -------- TRAIN --------
    model.train()
    for epoch in range(10):   # keep small initially
        total_loss = 0
        for imgs, txts, y in train_loader:
            imgs = imgs.to(device)
            txts = txts.to(device)
            y = y.to(device)

            out = model(imgs, txts)
            loss = tube_loss(y, out[:,0], out[:,1], t=t, r=r)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")



===== Training M3 for category: bikes =====


In [24]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "phones"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: phones =====
[phones] Epoch 1 | Loss: 0.1037
[phones] Epoch 2 | Loss: 0.0441
[phones] Epoch 3 | Loss: 0.0392
[phones] Epoch 4 | Loss: 0.0406
[phones] Epoch 5 | Loss: 0.0292


In [25]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["phones"]["log_mean"]
sigma = category_norm["phones"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (phones)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (phones)
PICP: 0.9333
MPIW: 75894.38


In [26]:
torch.save(
    {
        "category": "phones",
        "model": "M3_OpenAI_CLIP",
        "t": 0.98,
        "PICP": 0.9333,
        "MPIW": 75894.38,
        "l": l_pred,
        "u": u_pred,
        "y": y_true
    },
    "M3_phones_final.pt"
)

print("Saved final M3 phones results")


Saved final M3 phones results


In [27]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "bikes"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: bikes =====
[bikes] Epoch 1 | Loss: 0.0889
[bikes] Epoch 2 | Loss: 0.0583
[bikes] Epoch 3 | Loss: 0.0473
[bikes] Epoch 4 | Loss: 0.0430
[bikes] Epoch 5 | Loss: 0.0344


In [29]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["bikes"]["log_mean"]
sigma = category_norm["bikes"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (bikes)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (bikes)
PICP: 0.9570
MPIW: 209307.64


In [30]:
torch.save(
    {
        "category": "bikes",
        "model": "M3_OpenAI_CLIP",
        "t": 0.97,
        "PICP": 0.9570,
        "MPIW": 209307.64,
        "l": l_pred,
        "u": u_pred,
        "y": y_true
    },
    "M3_bikes_final.pt"
)

print("Saved final M3 bikes results")


Saved final M3 bikes results


In [31]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "books"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: books =====
[books] Epoch 1 | Loss: 0.1157
[books] Epoch 2 | Loss: 0.0345
[books] Epoch 3 | Loss: 0.0300
[books] Epoch 4 | Loss: 0.0290
[books] Epoch 5 | Loss: 0.0253


In [33]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["books"]["log_mean"]
sigma = category_norm["books"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (books)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (books)
PICP: 0.9348
MPIW: 12345.31


In [34]:
torch.save(
    {
        "category": "books",
        "model": "M3_OpenAI_CLIP",
        "t": 0.97,
        "PICP": 0.9348,
        "MPIW": 12345.31,
        "l": l_pred,
        "u": u_pred,
        "y": y_true
    },
    "M3_books_final.pt"
)

print("Saved final M3 books results")


Saved final M3 books results


In [35]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "cars"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: cars =====
[cars] Epoch 1 | Loss: 0.0952
[cars] Epoch 2 | Loss: 0.0609
[cars] Epoch 3 | Loss: 0.0543
[cars] Epoch 4 | Loss: 0.0494
[cars] Epoch 5 | Loss: 0.0470


In [36]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["cars"]["log_mean"]
sigma = category_norm["cars"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (cars)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (cars)
PICP: 1.0000
MPIW: 3222647.25


In [37]:
torch.save(
    {
        "category": "cars",
        "model": "M3_OpenAI_CLIP",
        "t": 0.97,
        "PICP": 1.0000,
        "MPIW": 3222647.25,
        "l": l_pred,
        "u": u_pred,
        "y": y_true
    },
    "M3_cars_final.pt"
)

print("Saved final M3 cars results")


Saved final M3 cars results


In [38]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "cycle"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: cycle =====
[cycle] Epoch 1 | Loss: 0.1346
[cycle] Epoch 2 | Loss: 0.0516
[cycle] Epoch 3 | Loss: 0.0483
[cycle] Epoch 4 | Loss: 0.0490
[cycle] Epoch 5 | Loss: 0.0423


In [39]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["cycle"]["log_mean"]
sigma = category_norm["cycle"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (cycle)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (cycle)
PICP: 0.9655
MPIW: 115418.27


In [40]:
torch.save(
    {
        "category": "cycle",
        "model": "M3_OpenAI_CLIP",
        "t": 0.97,
        "PICP": 0.9655,
        "MPIW": 115418.27,
        "l": l_pred,
        "u": u_pred,
        "y": y_true
    },
    "M3_cycle_final.pt"
)

print("Saved final M3 cycle results")


Saved final M3 cycle results


In [54]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "flat"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: flat =====
[flat] Epoch 1 | Loss: 0.1257
[flat] Epoch 2 | Loss: 0.0587
[flat] Epoch 3 | Loss: 0.0356
[flat] Epoch 4 | Loss: 0.0309
[flat] Epoch 5 | Loss: 0.0211


In [55]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["flat"]["log_mean"]
sigma = category_norm["flat"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (flat)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (flat)
PICP: 1.0000
MPIW: 22194196.00


In [56]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "fridges"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: fridges =====
[fridges] Epoch 1 | Loss: 0.0875
[fridges] Epoch 2 | Loss: 0.0427
[fridges] Epoch 3 | Loss: 0.0386
[fridges] Epoch 4 | Loss: 0.0351
[fridges] Epoch 5 | Loss: 0.0344


In [57]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["fridges"]["log_mean"]
sigma = category_norm["fridges"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (fridges)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (fridges)
PICP: 0.9867
MPIW: 75373.81


In [58]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "games"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: games =====
[games] Epoch 1 | Loss: 0.6580
[games] Epoch 2 | Loss: 0.3772
[games] Epoch 3 | Loss: 0.2416
[games] Epoch 4 | Loss: 0.1342
[games] Epoch 5 | Loss: 0.0332


In [59]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["games"]["log_mean"]
sigma = category_norm["games"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (games)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (games)
PICP: 0.5000
MPIW: 37483.30


In [60]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "gamesentertainment"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: gamesentertainment =====
[gamesentertainment] Epoch 1 | Loss: 0.0737
[gamesentertainment] Epoch 2 | Loss: 0.0327
[gamesentertainment] Epoch 3 | Loss: 0.0316
[gamesentertainment] Epoch 4 | Loss: 0.0297
[gamesentertainment] Epoch 5 | Loss: 0.0280


In [61]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["gamesentertainment"]["log_mean"]
sigma = category_norm["gamesentertainment"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (gamesentertainment)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (gamesentertainment)
PICP: 0.9600
MPIW: 51257.56


In [62]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "laptop"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: laptop =====
[laptop] Epoch 1 | Loss: 0.0794
[laptop] Epoch 2 | Loss: 0.0407
[laptop] Epoch 3 | Loss: 0.0360
[laptop] Epoch 4 | Loss: 0.0325
[laptop] Epoch 5 | Loss: 0.0311


In [63]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["laptop"]["log_mean"]
sigma = category_norm["laptop"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (laptop)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (laptop)
PICP: 0.9762
MPIW: 74093.16


In [64]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "mobile"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: mobile =====
[mobile] Epoch 1 | Loss: 0.0826
[mobile] Epoch 2 | Loss: 0.0310
[mobile] Epoch 3 | Loss: 0.0336
[mobile] Epoch 4 | Loss: 0.0310
[mobile] Epoch 5 | Loss: 0.0265


In [65]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["mobile"]["log_mean"]
sigma = category_norm["mobile"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (mobile)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (mobile)
PICP: 0.9444
MPIW: 84001.84


In [71]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "printer"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: printer =====
[printer] Epoch 1 | Loss: 0.1124
[printer] Epoch 2 | Loss: 0.0385
[printer] Epoch 3 | Loss: 0.0357
[printer] Epoch 4 | Loss: 0.0309
[printer] Epoch 5 | Loss: 0.0304


In [72]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["printer"]["log_mean"]
sigma = category_norm["printer"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (printer)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (printer)
PICP: 0.9500
MPIW: 457610.84


In [73]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "tv"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: tv =====
[tv] Epoch 1 | Loss: 0.0972
[tv] Epoch 2 | Loss: 0.0506
[tv] Epoch 3 | Loss: 0.0421
[tv] Epoch 4 | Loss: 0.0371
[tv] Epoch 5 | Loss: 0.0337


In [74]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["tv"]["log_mean"]
sigma = category_norm["tv"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (tv)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (tv)
PICP: 0.7917
MPIW: 29977.03


In [75]:
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch

results_m3 = {}

# 🚨 TRAIN ONE CATEGORY AT A TIME (change later)
cat = "washingmachine"
print(f"\n===== Training M3 for category: {cat} =====")

train_ds = category_datasets[cat]["train"]
val_ds   = category_datasets[cat]["val"]
test_ds  = category_datasets[cat]["test"]

# 🔽 MEMORY-SAFE DATALOADERS
train_loader = DataLoader(
    train_ds,
    batch_size=4,          # 🔑 reduced
    shuffle=True,
    num_workers=0          # 🔑 Windows-safe
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    num_workers=0
)

# ✅ MODEL
model = CLIPPIModel(clip_model).to(device)

# 🚀 TRAIN ONLY THE PI HEAD
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.98
r = 0.5

# -------- TRAIN --------
model.train()

for epoch in range(5):   # 🔽 fewer epochs for stability
    total_loss = 0

    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:, 0], out[:, 1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[{cat}] Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 🧹 CLEAN MEMORY
    torch.cuda.empty_cache()



===== Training M3 for category: washingmachine =====
[washingmachine] Epoch 1 | Loss: 0.0932
[washingmachine] Epoch 2 | Loss: 0.0449
[washingmachine] Epoch 3 | Loss: 0.0410
[washingmachine] Epoch 4 | Loss: 0.0382
[washingmachine] Epoch 5 | Loss: 0.0353


In [76]:
import torch

model.eval()

y_true = []
l_pred = []
u_pred = []

# category-specific normalization params
mu = category_norm["washingmachine"]["log_mean"]
sigma = category_norm["washingmachine"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        # ensure correct ordering
        l = torch.min(out[:, 0], out[:, 1])
        u = torch.max(out[:, 0], out[:, 1])

        # denormalize (log-space → price)
        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

# concatenate
l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

# -------- METRICS --------
picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM3 — OpenAI CLIP (washingmachine)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")



M3 — OpenAI CLIP (washingmachine)
PICP: 0.9859
MPIW: 29067.42


In [50]:
import torch
import pandas as pd

model.eval()

results = {}

ALL_CATEGORIES = [
    "bikes", "books", "cars", "cycle", "phones",
    "flat", "fridges", "games", "gamesentertainment",
    "laptop", "mobile", "printer", "tv", "washingmachine"
]

with torch.no_grad():
    for cat in ALL_CATEGORIES:
        print(f"\nEvaluating M3 — {cat}")

        test_ds = category_datasets[cat]["test"]

        test_loader = DataLoader(
            test_ds,
            batch_size=4,      # memory-safe
            num_workers=0
        )

        mu = category_norm[cat]["log_mean"]
        sigma = category_norm[cat]["log_std"]

        y_true, l_pred, u_pred = [], [], []

        for imgs, txts, y in test_loader:
            imgs = imgs.to(device)
            txts = txts.to(device)
            y = y.to(device)

            out = model(imgs, txts)

            l = torch.min(out[:, 0], out[:, 1])
            u = torch.max(out[:, 0], out[:, 1])

            # denormalize (log → price)
            l = torch.exp(l * sigma + mu)
            u = torch.exp(u * sigma + mu)
            y = torch.exp(y * sigma + mu)

            l_pred.append(l.cpu())
            u_pred.append(u.cpu())
            y_true.append(y.cpu())

        l_pred = torch.cat(l_pred)
        u_pred = torch.cat(u_pred)
        y_true = torch.cat(y_true)

        picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean().item()
        mpiw = (u_pred - l_pred).mean().item()

        results[cat] = {
            "PICP": picp,
            "MPIW": mpiw
        }

        # free memory aggressively
        del l_pred, u_pred, y_true
        torch.cuda.empty_cache()

# ---------- SUMMARY TABLE ----------
df_results = pd.DataFrame(results).T
print("\n===== M3 CATEGORY-WISE RESULTS =====")
print(df_results)

# ---------- OPTIONAL SAVE ----------
torch.save(results, "M3_all_categories_results.pt")
df_results.to_csv("M3_all_categories_results.csv")

print("\nSaved M3 results to disk")



Evaluating M3 — bikes

Evaluating M3 — books

Evaluating M3 — cars

Evaluating M3 — cycle

Evaluating M3 — phones

Evaluating M3 — flat

Evaluating M3 — fridges

Evaluating M3 — games

Evaluating M3 — gamesentertainment

Evaluating M3 — laptop

Evaluating M3 — mobile

Evaluating M3 — printer

Evaluating M3 — tv

Evaluating M3 — washingmachine

===== M3 CATEGORY-WISE RESULTS =====
                        PICP          MPIW
bikes               0.903226  3.094288e+05
books               0.891304  5.373931e+03
cars                0.848485  1.951718e+06
cycle               0.862069  2.269200e+04
phones              0.866667  7.347808e+04
flat                0.980392  4.451048e+07
fridges             0.920000  2.517604e+04
games               0.500000  3.868403e+04
gamesentertainment  0.933333  6.143466e+04
laptop              0.912698  6.712880e+04
mobile              0.888889  7.104781e+04
printer             1.000000  7.697923e+04
tv                  0.902778  4.091757e+04
washingmachine

In [66]:
import open_clip
import torch

CLIP_MODEL_NAME = "ViT-H-14"

clip_model_m4, clip_preprocess, _ = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained="laion2b_s32b_b79k"
)

clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)

clip_model_m4 = clip_model_m4.to(device)
clip_model_m4.eval()

for p in clip_model_m4.parameters():
    p.requires_grad = False

print("LAION CLIP ViT-H/14 loaded & frozen")


LAION CLIP ViT-H/14 loaded & frozen


In [67]:
import torch.nn as nn

class CLIPPIModel_M4(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model
        embed_dim = clip_model.text_projection.shape[1]  # 1024 for ViT-H/14

        self.head = nn.Sequential(
            nn.Linear(embed_dim * 2, 512),
            nn.ReLU(),
            nn.Linear(512, 2)  # lower, upper
        )

    def forward(self, images, texts):
        img_feat = self.clip.encode_image(images)
        txt_feat = self.clip.encode_text(texts)
        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.head(fused)


In [68]:
from torch.utils.data import DataLoader

cat = "phones"

train_ds = category_datasets[cat]["train"]
test_ds  = category_datasets[cat]["test"]

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, num_workers=0
)

test_loader = DataLoader(
    test_ds, batch_size=4, num_workers=0
)


In [69]:
from torch.optim import Adam

model = CLIPPIModel_M4(clip_model_m4).to(device)
optimizer = Adam(model.head.parameters(), lr=1e-4)

t = 0.95
r = 0.5

model.train()

for epoch in range(5):
    total_loss = 0
    for imgs, txts, y in train_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)
        loss = tube_loss(y, out[:,0], out[:,1], t=t, r=r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[phones][M4] Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")
    torch.cuda.empty_cache()


[phones][M4] Epoch 1 | Loss: 0.1276
[phones][M4] Epoch 2 | Loss: 0.0860
[phones][M4] Epoch 3 | Loss: 0.0680
[phones][M4] Epoch 4 | Loss: 0.0580
[phones][M4] Epoch 5 | Loss: 0.0584


In [70]:
model.eval()

y_true, l_pred, u_pred = [], [], []

mu = category_norm["phones"]["log_mean"]
sigma = category_norm["phones"]["log_std"]

with torch.no_grad():
    for imgs, txts, y in test_loader:
        imgs = imgs.to(device)
        txts = txts.to(device)
        y = y.to(device)

        out = model(imgs, txts)

        l = torch.min(out[:,0], out[:,1])
        u = torch.max(out[:,0], out[:,1])

        l = torch.exp(l * sigma + mu)
        u = torch.exp(u * sigma + mu)
        y = torch.exp(y * sigma + mu)

        l_pred.append(l.cpu())
        u_pred.append(u.cpu())
        y_true.append(y.cpu())

l_pred = torch.cat(l_pred)
u_pred = torch.cat(u_pred)
y_true = torch.cat(y_true)

picp = ((y_true >= l_pred) & (y_true <= u_pred)).float().mean()
mpiw = (u_pred - l_pred).mean()

print("\nM4 — LAION CLIP (phones)")
print(f"PICP: {picp.item():.4f}")
print(f"MPIW: {mpiw.item():.2f}")
 


M4 — LAION CLIP (phones)
PICP: 0.8444
MPIW: 49250.34
